# Privacy & Governance

### Data Privacy, Governance, and Compliance Assessment  

The workflow includes the following key components:

1. **Identification of Personally Identifiable Information (PII)**
   - Detects sensitive fields such as names, emails, social security numbers, IP addresses, and dates of birth.

2. **Pseudonymization and Hashing**
   - Protects sensitive data through pseudonymization or hashing, reducing privacy risks while preserving analytical utility.
   - Maintains **audit logs** for all changes to ensure traceability and accountability.

3. **Consent Tracking**
   - Records user consent for data processing, including purpose, method, and timestamp, supporting GDPR compliance.

4. **Data Retention and Storage Limitation**
   - Implements automatic expiration of records via TTL indexes, ensuring that personal data is not stored longer than necessary.

5. **Audit Trails and Oversight**
   - Captures logs of decisions and human oversight for high-risk processes, such as credit scoring.

6. **Regulatory Mapping**
   - Maps dataset fields to relevant **GDPR articles** and references the **EU AI Act** classification for high-risk AI systems, such as credit scoring models.

7. **Automated Compliance Reporting**
   - Generates a detailed report highlighting potential privacy and governance gaps, along with mitigation status and health scores.

8. **Actionable Governance Controls**

The goal of this notebook is to demonstrate **best practices for privacy, data governance, and regulatory compliance** in a structured, auditable, and reproducible way, suitable for both academic analysis and real-world implementation.

## 0. Data Loading

In [4]:
# Install pymongo
!pip install pymongo
!pip install pandas
!pip install faker  

In [5]:
from pymongo import MongoClient
import json
import os

# Connect to MongoDB
client = MongoClient("mongodb://localhost:27017/")  # Adjust URI if needed
db = client["credit_applications_db"]
collection = db["applications"]


# Load JSON data
file_path = "cleaned_credit_applications.json"  # Must be in cwd
if not os.path.exists(file_path):
    raise FileNotFoundError(f"JSON file not found at {file_path}")

with open(file_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} records from JSON.")

# Drop collection to start fresh
db.drop_collection("applications")


# Ensure uniqueness of documents in the collection before re-inserting/updating
seen_ids = set()
unique_documents = []

# Fetch all documents currently in the collection
for doc in collection.find():
    if '_id' in doc:
        if doc['_id'] not in seen_ids:
            unique_documents.append(doc)
            seen_ids.add(doc['_id'])
        else:
            print(f"Skipping duplicate document with _id: {doc['_id']} already in the collection.")

# Insert only unique documents
if unique_documents:
    try:
        collection.insert_many(unique_documents)
        print(f"Dataset loaded: {collection.count_documents({})} records ready for audit.\n")
    except Exception as e:
        print(f"Error inserting documents: {e}")
else:
    print("No documents to insert after removing duplicates.\n")

# Insert JSON into MongoDB
collection.insert_many(data)
print(f"Inserted {collection.count_documents({})} documents into MongoDB")

Loaded 500 records from JSON.
No documents to insert after removing duplicates.

Inserted 500 documents into MongoDB


In [6]:
sample = collection.find_one()
print("Sample document keys:", sample.keys())

Sample document keys: dict_keys(['_id', 'spending_behavior', 'processing_timestamp', 'loan_purpose', 'notes', 'applicant_info_full_name', 'applicant_info_email', 'applicant_info_ssn', 'applicant_info_ip_address', 'applicant_info_gender', 'applicant_info_date_of_birth', 'applicant_info_zip_code', 'financials_annual_income', 'financials_credit_history_months', 'financials_debt_to_income', 'financials_savings_balance', 'decision_loan_approved', 'decision_rejection_reason', 'decision_interest_rate', 'decision_approved_amount', 'financials_annual_salary', 'financials_annual_income_canonical', 'analysis_dob_format', 'analysis_dob_parsed', 'analysis_age_years'])


## 1. PII Audit Report Generation (Pre-Pseudonymization)

This section of code performs a **comprehensive audit of personal data** in the collection before any pseudonymization or hashing is applied. It follows these steps:

1. **Define PII Inventory**  
   - Categorizes fields into **Direct Identifiers**, **Online Identifiers**, and **Quasi-Identifiers / Demographic**.  
   - Examples include names, emails, SSNs, IP addresses, date of birth, zip codes, and gender.

2. **Scan the Collection**  
   - For each field, counts how many documents contain that data and are not empty or null.  
   - Fetches one example value from the collection as evidence of exposure.

3. **Store Results**  
   - Creates a list of dictionaries with audit details:
     - Category of PII  
     - Field name  
     - Number of exposed records  
     - Total records in the collection  
     - Example value (or mark as secure/missing)

**Purpose:**  
- This audit allows us to **identify which personal data is still exposed**, providing a baseline before implementing pseudonymization, hashing, and GDPR-compliant processing.  

In [7]:
import pandas as pd

# Updated PII inventory matching your document fields
pii_inventory = {
    "Direct Identifiers": [
        "applicant_info_full_name",
        "applicant_info_email",
        "applicant_info_ssn"
    ],
    "Online Identifiers": [
        "applicant_info_ip_address"
    ],
    "Quasi-Identifiers / Demographic": [
        "applicant_info_date_of_birth",
        "applicant_info_zip_code",
        "applicant_info_gender"
    ]
}

# Prepare a list to store the audit results
audit_results = []

# Total records in the collection
total_records = collection.count_documents({})

# Scan for PII exposures
for category, fields in pii_inventory.items():
    for field in fields:
        # Query documents where field exists and is not empty/null
        query = {field: {"$exists": True, "$nin": ["", None]}}
        exposure_count = collection.count_documents(query)

        if exposure_count > 0:
            # Fetch one example for evidence
            example_doc = collection.find_one(query, {"_id": 1, field: 1})
            exposed_value = example_doc.get(field, "N/A")

            audit_results.append({
                "Category": category,
                "Field": field,
                "Exposed Records": exposure_count,
                "Total Records": total_records,
                "Evidence (Example)": exposed_value
            })
        else:
            audit_results.append({
                "Category": category,
                "Field": field,
                "Exposed Records": 0,
                "Total Records": total_records,
                "Evidence (Example)": "Secure / Missing"
            })

# Convert results to a DataFrame for a clean table output
pii_report_df = pd.DataFrame(audit_results)

# Display the PII audit report
print("___________________________________________________")
print("PII Identification Report (Pre-Pseudonymization)")
print("___________________________________________________\n")
pii_report_df

___________________________________________________
PII Identification Report (Pre-Pseudonymization)
___________________________________________________



,Category,Field,Exposed Records,Total Records,Evidence (Example)
0,Direct Identifiers,applicant_info_full_name,500,500,Jerry Smith
1,Direct Identifiers,applicant_info_email,489,500,jerry.smith17@hotmail.com
2,Direct Identifiers,applicant_info_ssn,496,500,596-64-4340
3,Online Identifiers,applicant_info_ip_address,496,500,192.168.48.155
4,Quasi-Identifiers / Demographic,applicant_info_date_of_birth,496,500,2001-03-09
5,Quasi-Identifiers / Demographic,applicant_info_zip_code,499,500,10036
6,Quasi-Identifiers / Demographic,applicant_info_gender,500,500,Male


The PII Identification Report shows that the dataset contains significant exposure of both direct and quasi-identifiers. The key practical implications are:

1. **High Exposure of Sensitive Data**  
   Fields such as `full_name`, `email`, `ssn`, `ip_address`, and `gender` are present in nearly all records. This means that unauthorized access to the dataset could directly identify individuals, posing a serious privacy and compliance risk.

2. **Risk of Re-identification via Quasi-Identifiers**  
   Attributes like `date_of_birth`, `zip_code`, and `gender` can be combined to indirectly identify individuals, even if direct identifiers are removed.

3. **GDPR Compliance Concerns**  
   According to GDPR, personal data must be minimized, protected, and processed lawfully. The current dataset requires:
   - Pseudonymization or anonymization of sensitive fields  
   - Consent tracking for data processing  
   - Audit trails and the ability to erase data upon request  

4. **Audit & Reporting Requirements**  
   Evidence of exposed PII (e.g., `Jerry Smith`, `jerry.smith17@hotmail.com`) must be documented in audit logs. This ensures transparency and regulatory accountability.

**In summary:**  
The dataset in its current form **cannot be shared or used safely**. Implementing pseudonymization, hashing, consent tracking, and audit logging is essential to protect individuals and ensure GDPR compliance.

### Identifying Potential PII Fields

This code scans a sample document to identify fields that may contain personally identifiable information (PII).  

**Explanation:**
- It iterates over all fields in the document.  
- For each field, it evaluates whether the content or the field name suggests it could be sensitive personal data.  
- Fields that are likely to contain PII are collected into a list, with duplicates removed.  
- The resulting list provides a quick overview of which parts of the dataset require special handling, such as anonymization or pseudonymization, to ensure privacy compliance.

In [8]:
pii_fields = []
for key, value in sample.items():
    if isinstance(value, str):
        if "@" in value:
            pii_fields.append(key)
        elif any(char.isdigit() for char in value) and "ssn" in key.lower():
            pii_fields.append(key)
        elif key.lower() in ["dob", "birth_date", "date_of_birth"]:
            pii_fields.append(key)
        elif "ip" in key.lower():
            pii_fields.append(key)
        elif key.lower() in ["first_name", "last_name", "full_name"]:
            pii_fields.append(key)

pii_fields = list(set(pii_fields))
print("Identified PII fields:", pii_fields)

Identified PII fields: ['applicant_info_ip_address', 'applicant_info_email', 'applicant_info_zip_code', 'applicant_info_ssn']


## 2. Pseudonymization and Hashing

### Pseudonymizing Personal Names

This code demonstrates how to pseudonymize sensitive data in the dataset:

- A function is defined to replace a real name with a randomly generated fake name.  
- The code iterates through all documents in the MongoDB collection.  
- For each document, the original name is replaced with a pseudonym, and the document is updated in the database.  
- This process masks the real identities of individuals while preserving the structure of the dataset for analysis.  

Pseudonymization helps **reduce privacy risks** and supports compliance with data protection regulations such as GDPR.

In [9]:
from faker import Faker
fake = Faker()

# Function to pseudonymize names
def pseudonymize_name(name):
    return fake.name()

# Update documents in MongoDB
for doc in collection.find():
    new_name = pseudonymize_name(doc.get("full_name", ""))
    collection.update_one({"_id": doc["_id"]}, {"$set": {"full_name": new_name}})

print("Pseudonymization of full_name completed.")

Pseudonymization of full_name completed.


### Pseudonymizing Remaining Sensitive Fields with Audit Logging

To further protect personally identifiable information (PII), the remaining sensitive fields, such as email, social security number, date of birth, and IP address are pseudonymized using a **hashing function**.  

Key points of this process:  
- Each field value is converted into a **hash**, making it irreversible while preserving uniqueness for analysis.  
- An **audit log** records every change, including:
  - The document ID  
  - The field name  
  - A partially masked version of the original value  
  - The new hashed value  
  - The timestamp of the change  
  - The type of action (pseudonymization)  
- This ensures **traceability** and accountability, which are important for regulatory compliance.  

By implementing this step, all sensitive PII in the dataset is protected, while maintaining an auditable record of modifications for governance and compliance purposes.

In [10]:
from hashlib import sha256

# Function to hash sensitive data
def hash_field(value):
    if value:
        return sha256(value.encode('utf-8')).hexdigest()
    return value

# Fields to pseudonymize/hash
sensitive_fields = ["email", "ssn", "dob", "ip_address"]

# Add audit log collection
audit_collection = db["audit_log"]

# Update each document
for doc in collection.find():
    updates = {}
    for field in sensitive_fields:
        if field in doc:
            original_value = doc[field]
            hashed_value = hash_field(original_value)
            updates[field] = hashed_value

            # Log the change in audit log
            audit_collection.insert_one({
                "document_id": doc["_id"],
                "field": field,
                "old_value": original_value[:4] + "***" if original_value else None,  # partial masking
                "new_value": hashed_value,
                "timestamp": datetime.utcnow(),
                "action": "pseudonymization"
            })

    if updates:
        collection.update_one({"_id": doc["_id"]}, {"$set": updates})

print("All remaining sensitive PII pseudonymized with audit logging.")

All remaining sensitive PII pseudonymized with audit logging.
